[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/belief_propagation/belief_propagation.ipynb)

# The Most Likely Poses of a Lap

A robot drives a lap and reads, at every stop, how far it moved since the last one. Each reading is a little off, so adding them up, dead reckoning, drifts. Passing a spot it saw on its first lap, it reads one more relative pose, and the lap closes. This notebook finds the poses most likely given all the readings, and how uncertain each one is, by belief propagation: every pose keeps a Gaussian belief, updated from what its readings tell it. Every step works on one pose or one reading at a time, never on one system for all poses at once.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from dataclasses import dataclass
from itertools import accumulate

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA2D
from examples.animation import save_animation
from examples.geometry.belief_propagation import render

np.set_printoptions(precision=4, suppress=True)

ga = PGA2D
mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
# A pose, and a small motion on the right of a pose.
Motor = ga.gatype.rotor()
Twist = ga.gatype.bivector()
# A line reads out a twist, `line & twist`; a plane reads out a point, `plane & point`.
Line = ga.gatype.antibivector()
Plane = ga.gatype.vector()
Information = ga.gatype((Line, Twist))
Covariance = ga.gatype((Twist, Line))
Quadric = ga.gatype((Plane, Point))
# The point each pose carries: the origin, dual to the weight direction w.
origin = mv.w.dual()                                                            # [] Point

## 1. The lap and its readings

The robot takes steps of 0.7 m and turns a lap every 36 poses, faster and slower twice a lap. Each step is read with an error: a twist on the right of the true step, drawn from the reading's covariance, a map from lines to twists. The information of a reading is the inverse of its covariance. Dead reckoning chains the readings of the steps from the known start. One more reading closes the loop: the last pose, as read from the pose one lap before it. That reading says the two sit a small step apart; dead reckoning has drifted them apart by much more.

In [ ]:
POSES, PER_LAP = 40, 36


def information(translation_std: float, rotation_std: float) -> Information:
    """The information of a reading with isotropic translation noise and rotation noise about the head."""
    translation = mv.yw * (mv.yw & Line) + mv.wx * (mv.wx & Line)               # [] Covariance
    rotation = mv.xy * (mv.xy & Line)                                           # [] Covariance
    return (translation * translation_std**2 + rotation * rotation_std**2).inverse()


def sample(covariance: Covariance, rng: np.random.Generator, count: int) -> Twist:
    """Twists drawn from a covariance: independent unit draws along lines orthonormal in its form."""
    spread = Line & covariance                                                  # [] Scalar <- (Line, Line)
    _, lines = spread.eigh(spread)                                              # [modes] Line
    return (covariance(lines) * rng.normal(size=(count,) + lines.shape)).sum(axis=-1)   # [count] Twist


# How far a reading is trusted, and how well the first pose is known. The other poses are known before
# any reading only vaguely, near where dead reckoning puts them.
reading = information(0.03, 0.01)                                               # [] Information
start = information(0.01, 0.01)                                                 # [] Information
vague = information(100.0, 30.0)                                                # [] Information

rng = np.random.default_rng(0)
phase = 2 * np.pi * np.arange(POSES - 1) / PER_LAP
steps = ((mv.xy * (1 + 0.4 * np.sin(2 * phase)) * 2 * np.pi / PER_LAP - mv.wx * 0.7) * 0.5).exp()   # [poses - 1] Motor
truth = stack(list(accumulate(steps, lambda pose, step: pose * step, initial=mv.rotor())))           # [poses] Motor
# Every step read, then the last pose read from the pose one lap before it; each head from its tail.
tails = np.append(np.arange(POSES - 1), POSES - 1)
heads = np.append(np.arange(1, POSES), POSES - 1 - PER_LAP)
readings = (truth[tails].inverse() * truth[heads]) * (sample(reading.inverse(), rng, POSES) * 0.5).exp()   # [readings] Motor
dead = stack(list(accumulate(readings[:-1], lambda pose, step: pose * step, initial=truth[0])))   # [poses] Motor
# The two poses the closing reading links, where dead reckoning puts them.
render.draw_lap(truth, dead, dead[[tails[-1], heads[-1]]]);

## 2. The most likely poses

A reading's mismatch is a twist: the log of the motor between what it read and the relative motor of the current poses, `(reading.inverse() * relative).log() * 2`. The most likely poses minimize half the sum of `mismatch & information(mismatch)` over the readings, and the same for each pose's offset from where it was believed to be before any reading. A reading is seen from both its ends. From its tail, the reading and the relative motor are inverted, and the information is carried into the tail's frame by the head's pose in it: `head >> information(head << Twist)`. The gradient with respect to the twist of every pose gathers `information(mismatch)` at both ends of every reading. Dead reckoning satisfies every reading of a step, so on the chain of steps the gradient vanishes there; the reading that closes the loop does not.

In the notation of nonlinear least squares on manifolds the mismatch reads as the residual $r_{ij} = \log\!\left(Z_{ij}^{-1} X_i^{-1} X_j\right)^\vee$ and the minimized sum as $\tfrac12 \sum r_{ij}^\top \Sigma_{ij}^{-1} r_{ij}$.

In [ ]:
@dataclass(frozen=True)
class PoseGraph:
    """Relative pose readings between pairs of poses, and a prior on every pose."""

    ends: np.ndarray          # [2, readings] the pose at the head of each reading, then the one at its tail
    readings: Motor           # [readings] Motor
    information: Information  # [readings] Information
    anchors: Motor            # [poses] Motor
    priors: Information       # [poses] Information

    @property
    def incidence(self) -> np.ndarray:
        """[2, readings, poses] one where an end of a reading sits at a pose."""
        return (self.ends[..., None] == np.arange(len(self.anchors))).astype(float)


def linearize(graph: PoseGraph, poses: Motor):
    """Every reading as seen from each of its ends: the near pose in the far pose's frame, the
    mismatch, and the information in the near pose's frame; and every pose's offset from its anchor."""
    relative = poses[graph.ends[::-1]].inverse() * poses[graph.ends]            # [2, readings] Motor
    measured = stack([graph.readings, graph.readings.inverse()])               # [2, readings] Motor
    mismatch = (measured.inverse() * relative).log() * 2                       # [2, readings] Twist
    head = relative[0]                                                         # [readings] Motor
    information = stack([graph.information, head >> graph.information(head << Twist)])   # [2, readings] Information
    offset = (graph.anchors.inverse() * poses).log() * 2                       # [poses] Twist
    return relative, mismatch, information, offset


def gradient(graph: PoseGraph, poses: Motor) -> Line:
    """The gradient of half the weighted squared mismatches and offsets, per pose."""
    _, mismatch, information, offset = linearize(graph, poses)
    return (information(mismatch)[..., None] * graph.incidence).sum(axis=(-3, -2)) + graph.priors(offset)   # [poses] Line


anchors = concatenate([truth[:1], dead[1:]])                                    # [poses] Motor
priors = concatenate([start[None], vague * np.ones(POSES - 1)])                 # [poses] Information
ends = np.stack([heads, tails])                                                 # [2, readings]
chain = PoseGraph(ends[:, :-1], readings[:-1], reading * np.ones(POSES - 1), anchors, priors)
loop = PoseGraph(ends, readings, reading * np.ones(POSES), anchors, priors)
chain_gradient = gradient(chain, dead)                                          # [poses] Line
loop_gradient = gradient(loop, dead)                                            # [poses] Line

In [ ]:
print("largest gradient at dead reckoning, chain:", np.abs(chain_gradient.kernel).max())
print("largest gradient at dead reckoning, loop: ", np.abs(loop_gradient.kernel).max())

## 3. Beliefs, and what a reading tells

Every pose holds a belief, a Gaussian over its twist: an information and an information-weighted mean, with mean `information.solve(weighted_mean)` and covariance `information.inverse()`. A reading tells each of its ends what the belief at its other end implies. That belief is taken without what the reading itself told it, and carried into the near pose's frame by the relative motor: the mean as `relative << mean`, the covariance as `relative << covariance(relative >> Line)`. Through the reading, the near pose sits its mismatch behind that mean, and the reading's own covariance adds to the carried one. A pose's belief is its prior plus all it is told. Every round the poses move to their beliefs' means, and what they were told moves with them.

On the chain, once what is told has crossed it, every belief is exact. The exact covariance can be found from means alone: nudge one pose's weighted mean by a line, and its most likely twist moves by its covariance applied to that line.

In the notation of Gaussian belief propagation the information reads as the information matrix $\Lambda$, the weighted mean as the information vector $\eta = \Lambda \mu$, and what a reading tells a pose as the factor-to-variable message, $\Lambda_{f \to i} = \Lambda_{ii} - \Lambda_{ij} \left(\Lambda_{jj} + \Lambda_{j \setminus f}\right)^{-1} \Lambda_{ji}$.

In [ ]:
@dataclass(frozen=True)
class Belief:
    """A Gaussian over the twist of a pose, as its information and its information-weighted mean."""

    information: Information  # [...] Information
    weighted_mean: Line       # [...] Line


def untold(graph: PoseGraph) -> Belief:
    """What the readings tell their ends before anything has been told: nothing."""
    return Belief(graph.information * np.zeros(graph.ends.shape), mv(Line, np.zeros(graph.ends.shape + (len(Line.output_subspace),))))


def beliefs(graph: PoseGraph, told: Belief, weighted_mean: Line) -> Belief:
    """Every pose's belief: its prior, with the given weighted mean, plus all it is told."""
    information = graph.priors + (told.information[..., None] * graph.incidence).sum(axis=(-3, -2))   # [poses] Information
    weighted_mean = weighted_mean + (told.weighted_mean[..., None] * graph.incidence).sum(axis=(-3, -2))   # [..., poses] Line
    return Belief(information, weighted_mean)


def tell(graph: PoseGraph, relative: Motor, mismatch: Twist, information: Information, belief: Belief, told: Belief) -> Belief:
    """What every reading tells each of its ends: the belief at its other end, without what the reading
    told that end, carried through the reading."""
    far = graph.ends[::-1]
    rest = belief.information[far] - told.information[::-1]                    # [2, readings] Information
    mean = rest.solve(belief.weighted_mean[..., far] - told.weighted_mean[..., ::-1, :])   # [..., 2, readings] Twist
    covariance = relative << rest.solve(relative >> Line)                  # [2, readings] Covariance
    implied = (information.inverse() + covariance).inverse()                   # [2, readings] Information
    return Belief(implied, implied((relative << mean) - mismatch))


def propagate(graph: PoseGraph, poses: Motor, rounds: int) -> Iterator[tuple[Motor, Information]]:
    """Belief propagation relinearized every round: every reading tells its ends what it implies, and
    every pose moves to its belief's mean."""
    told = untold(graph)
    for _ in range(rounds):
        relative, mismatch, information, offset = linearize(graph, poses)
        told = tell(graph, relative, mismatch, information, beliefs(graph, told, -graph.priors(offset)), told)
        belief = beliefs(graph, told, -graph.priors(offset))
        mean = belief.information.solve(belief.weighted_mean)                  # [poses] Twist
        poses = poses * (mean * 0.5).exp()
        told = Belief(told.information, told.weighted_mean - told.information(mean[graph.ends]))
        yield poses, belief.information


def exact_covariance(graph: PoseGraph, poses: Motor, lines: Line, rounds: int) -> Twist:
    """Each pose's exact covariance applied to each line: how far its most likely twist moves when its
    own weighted mean alone is nudged by the line, with every mismatch zero."""
    relative, mismatch, information, _ = linearize(graph, poses)
    nudged = np.arange(len(graph.anchors))
    nudges = lines[:, None, None] * np.eye(len(nudged))                        # [lines, nudged, poses] Line
    told = untold(graph)
    for _ in range(rounds):
        told = tell(graph, relative, 0 * mismatch, information, beliefs(graph, told, nudges), told)
    belief = beliefs(graph, told, nudges)
    moved = belief.information.solve(belief.weighted_mean)                     # [lines, nudged, poses] Twist
    return moved[:, nudged, nudged]                                            # [lines, poses] Twist


rounds = 400
chain_poses, chain_information = map(stack, zip(*propagate(chain, dead, rounds)))   # [rounds, poses] Motor, Information
basis = mv(Line, np.eye(len(Line.output_subspace)))                             # [lines] Line
chain_exact = exact_covariance(chain, chain_poses[-1], basis, POSES)            # [lines, poses] Twist
chain_believed = chain_information[-1].solve(basis[:, None])               # [lines, poses] Twist

In [ ]:
print("largest difference of believed and exact covariance on the chain:", np.abs((chain_believed - chain_exact).kernel).max())

## 4. Where each pose may be

A pose's covariance gives an ellipse around the point it carries. A twist moves the point by its commutator with the point, and reading that motion with a plane is reading the twist with a line, found by solving the incidence form; so the covariance of the twist gives the covariance of the point's motion. Added to the point's dyad, it is the point's second moment, a map from planes to points. Its inverse, paired twice with a point of unit weight, is one plus the squared number of standard deviations to it; less five weight dyads, it is a quadric that vanishes two standard deviations out. With odometry alone the ellipses grow away from the known start.

In [ ]:
def position_quadric(poses: Motor, covariance: Covariance, sigmas: float) -> Quadric:
    """The quadric sigmas standard deviations out, within which each pose carries the origin."""
    here = poses >> origin                                                     # [...] Point
    shift = Twist.commutator(here)(poses >> Twist)                             # [...] Point <- Twist
    readout = (Line & Twist).solve(Plane & shift)                              # [...] Line <- Plane
    moment = here * (Plane & here) + shift(covariance(readout))                # [...] Point <- Plane
    return moment.inverse() - (1 + sigmas**2) * mv.w * (mv.w & Point)          # [...] Plane <- Point


chain_ellipses = position_quadric(chain_poses, chain_information.inverse(), 2.0)   # [rounds, poses] Quadric
render.draw_survey({"odometry alone": (truth, dead, chain_poses, chain_ellipses)});

## 5. Closing the loop

With the reading that closes the loop, the poses move over the rounds from dead reckoning to the most likely ones, where the gradient vanishes. The means settle exactly; the covariances do not. What a reading tells goes round the loop and comes back to be counted again, and the believed variances differ from the exact ones. The animation shows the rounds on a logarithmic clock: with odometry alone the ellipses appear as what is told reaches each pose; closing the loop pulls the poses onto the lap and shrinks the ellipses.

In [ ]:
loop_poses, loop_information = map(stack, zip(*propagate(loop, dead, rounds)))   # [rounds, poses] Motor, Information
settled_gradient = gradient(loop, loop_poses[-1])                              # [poses] Line
loop_exact = exact_covariance(loop, loop_poses[-1], basis, rounds)              # [lines, poses] Twist
loop_believed = loop_information[-1].solve(basis[:, None])                  # [lines, poses] Twist
# The believed over the exact variance of each pose, along each basis line.
ratios = (basis[:, None] & loop_believed) / (basis[:, None] & loop_exact)       # [lines, poses] Scalar
loop_ellipses = position_quadric(loop_poses, loop_information.inverse(), 2.0)   # [rounds, poses] Quadric
runs = {"odometry alone": (truth, dead, chain_poses, chain_ellipses), "loop closed": (truth, dead, loop_poses, loop_ellipses)}
display(Image(filename=save_animation(render.animate_survey(runs, 60), "belief_propagation", 80)))

In [ ]:
print("largest gradient, at dead reckoning and settled:", np.abs(loop_gradient.kernel).max(), np.abs(settled_gradient.kernel).max())
print("believed over exact variance:", ratios.to_array().min(), "to", ratios.to_array().max())

In [ ]:
# checks
# On the chain dead reckoning is the most likely and every belief's covariance is exact; on the loop
# the gradient has fallen a millionfold.
np.testing.assert_allclose(chain_gradient.kernel, 0.0, atol=1e-8)
np.testing.assert_allclose((chain_believed - chain_exact).kernel, 0.0, atol=1e-10)
np.testing.assert_allclose(settled_gradient.kernel, 0.0, atol=1e-6 * np.abs(loop_gradient.kernel).max())